In [1]:

%pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 48.0 MB/s eta 0:00:00


In [13]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [14]:
def clean_data(data):

  data["pIC50"] = data["pIC50"].fillna(data["pIC50"].mean())
  data = data[data["num_atoms"]<60].copy()
  return data

In [15]:
def validate_smiles(data):

  valid_rows = []
  valid_count = 0
  invalid_count = 0
  for i, smile in enumerate (data["SMILES"]):
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
      invalid_count += 1
    else:
      valid_count += 1
      valid_rows.append(data.iloc[i])

  df_valid_rows = pd.DataFrame(valid_rows)
  print(f"Valid SMILES: {valid_count}, Invalid SMILES: {invalid_count}")

  return df_valid_rows

In [16]:
def generate_descriptors(data):
  weight=[]
  atoms=[]
  bonds=[]
  tpsa=[]
  numhdoner=[]
  numhaccept=[]

  for i in data["SMILES"]:
    mol = Chem.MolFromSmiles(i)
    weight_value = Descriptors.MolWt(mol)
    atoms_value = mol.GetNumAtoms()
    bonds_value = mol.GetNumBonds()
    tpsa_value = Descriptors.TPSA(mol)
    numhdoner_value = Descriptors.NumHDonors(mol)
    numhaccept_value = Descriptors.NumHAcceptors(mol)
    weight.append(weight_value)
    atoms.append(atoms_value)
    bonds.append(bonds_value)
    tpsa.append(tpsa_value)
    numhdoner.append(numhdoner_value)
    numhaccept.append(numhaccept_value)
  data["Molecular_Weight"] = weight
  data["Num_Atoms"] = atoms
  data["Num_Bonds"] = bonds
  data["TPSA"] = tpsa
  data["NumHDoner"] = numhdoner
  data["NumHAcceptor"] = numhaccept
  return data

In [17]:
def train_and_evaluate(data):
  x = data.drop(["SMILES","pIC50","mol"],axis=1)
  y = data["pIC50"]
  x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=20
  )
  models = {
    "linear_regression :" : LinearRegression(),
    "decision_tree_regressor :" : DecisionTreeRegressor(),
    "random_forest_regressor :" : RandomForestRegressor()
  }
  for name, model in models.items():
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    print(name)
    print("mean_absolute_error :",mean_absolute_error(y_test,y_pred))
    print("mean_squared_error :",mean_squared_error(y_test,y_pred))
    print("r2_score :",r2_score(y_test,y_pred))
  return data

In [18]:
def run_pipeline(filepath):
  data = pd.read_csv(filepath)
  data = clean_data(data)
  data = validate_smiles(data)
  data = generate_descriptors(data)
  data = train_and_evaluate(data)
  return data

run_pipeline(r"/content/SMILES_Big_Data_Set.csv")

Valid SMILES: 16067, Invalid SMILES: 0
linear_regression :
mean_absolute_error : 1.3262702166121252
mean_squared_error : 3.588607919987818
r2_score : 0.3590541164915092
decision_tree_regressor :
mean_absolute_error : 0.3697067810277345
mean_squared_error : 2.2243165566421035
r2_score : 0.6027243509498867
random_forest_regressor :
mean_absolute_error : 0.3507695294263622
mean_squared_error : 1.127269717629789
r2_score : 0.7986631860520876


,SMILES,pIC50,mol,num_atoms,logP,Molecular_Weight,Num_Atoms,Num_Bonds,TPSA,NumHDoner,NumHAcceptor
0,O=S(=O)(Nc1cccc(-c2cnc3ccccc3n2)c1)c1cccs1,4.26,<rdkit.Chem.rdchem.Mol object at 0x7f59df45bc30>,25,4.15910,367.455,25,28,71.95,1,5
1,O=c1cc(-c2nc(-c3ccc(-c4cn(CCP(=O)(O)O)nn4)cc3)...,4.34,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c9e0>,36,3.67430,506.434,36,40,149.78,4,5
2,NC(=O)c1ccc2c(c1)nc(C1CCC(O)CC1)n2CCCO,4.53,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cac0>,23,1.53610,317.389,23,25,101.37,3,4
3,NCCCn1c(C2CCNCC2)nc2cc(C(N)=O)ccc21,4.56,<rdkit.Chem.rdchem.Mol object at 0x7f59a320cba0>,22,0.95100,301.394,22,24,98.96,3,4
4,CNC(=S)Nc1cccc(-c2cnc3ccccc3n2)c1,4.59,<rdkit.Chem.rdchem.Mol object at 0x7f59a320c7b0>,21,3.21300,294.383,21,23,49.84,2,3
...,...,...,...,...,...,...,...,...,...,...,...
16082,S=C(NN=C(c1ccccn1)c1ccccn1)Nc1ccccc1,0.00,<rdkit.Chem.rdchem.Mol object at 0x7f59a314ed50>,24,3.21560,333.420,24,26,62.20,2,4
16083,S=C=NCCCCCCCCCCc1ccccc1,0.00,<rdkit.Chem.rdchem.Mol object at 0x7f59a314edc0>,19,5.45270,275.461,19,19,12.36,0,2
16084,S=C=NCCCCCCCCc1ccccc1,0.00,<rdkit.Chem.rdchem.Mol object at 0x7f59a314ee30>,17,4.67250,247.407,17,17,12.36,0,2
16085,S=c1[nH]nc(Cn2ccc3ccccc32)n1-c1ccccc1,0.00,<rdkit.Chem.rdchem.Mol object at 0x7f59a314eea0>,22,3.93289,306.394,22,25,38.54,1,2
